# 12. Does the stronger anonymization change the results?

The pipeline was first run with an anonymization made of six rules (outputs kept in `rules_only/`). Here the
regressors of step 7 on the GLiNER text are compared with the same regressors on the rules text: same outlets, folds,
settings and draws of k articles; paired bootstrap over outlets (1,000, seed 7).

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import boost as BO
import boost_eval as BE

art, outlets, rows = BO.load()
NEW = C.PREDICTIONS / "regression"
OLD = C.OUT / "rules_only" / "predictions" / "regression"
MODELS = ["TF-IDF", "Qwen3-8B", "Mistral-24B", "ModernBERT"]
done = [m for m in MODELS if len(list((NEW / m).glob("fold*.csv.gz"))) == 5 and len(list((OLD / m).glob("fold*.csv.gz"))) == 5]
print("compared:", done)

In [ ]:
def matrix(folder, name, k):
    p = pd.concat([pd.read_csv(f) for f in sorted((folder / name).glob("fold*.csv.gz"))]).set_index("article_id").prediction
    p = p.reindex(art.article_id).to_numpy()
    draws = [d for f in range(1, 6) for d in BO.test_draws(outlets, rows, f) if d[1] == str(k)]
    t = pd.DataFrame([(d, j, p[b].mean()) for d, _, j, b in draws], columns=["domain", "draw", "prediction"])
    return t.pivot_table(index="draw", columns="domain", values="prediction").reindex(columns=outlets.index).to_numpy()


idx = np.vstack([np.arange(len(outlets))[None, :], np.random.default_rng(7).integers(0, len(outlets), (BE.B, len(outlets)))])
out = []
for m in done:
    for k in ["1", "5", "all"]:
        a = BE._one(matrix(OLD, m, k), m, k, outlets, idx)[2]
        b = BE._one(matrix(NEW, m, k), m, k, outlets, idx)[2]
        for (score, mes), v in b.items():
            d = v - a[(score, mes)]; lo, hi = np.percentile(d[1:], [2.5, 97.5])
            out.append({"model": m, "k": k, "score": score, "measure": mes, "rules": a[(score, mes)][0], "GLiNER": v[0],
                        "difference": d[0], "low": lo, "high": hi,
                        "better": bool(hi < 0) if mes in BE.LOWER else bool(lo > 0), "worse": bool(lo > 0) if mes in BE.LOWER else bool(hi < 0)})
T = pd.DataFrame(out); T.to_csv(C.OUT / "anonymization_effect.csv", index=False)

In [ ]:
def table(k="all", score="stretched"):
    d = T[(T.k == k) & (T.score == score)].copy()
    f = lambda v, m: f"{v:.3f}" if m in ("Spearman", "Kendall") else f"{v:.1f}"
    d["text"] = [f"{f(a, m)} → {f(b, m)}" + (" ✓" if g else " ✗" if w else "") for a, b, m, g, w in
                 zip(d["rules"], d["GLiNER"], d.measure, d.better, d.worse)]
    return d.pivot_table(index="model", columns="measure", values="text", aggfunc="first")[BE.MEASURES].reindex(done)


table("all", "stretched")

In [ ]:
table("all", "normal")

In [ ]:
table("5", "stretched")

In [ ]:
table("1", "stretched")